In [1]:
print("all okay")

all okay


In [19]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings


In [2]:
load_dotenv()

True

In [5]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print("ENV VAR LOADED")


ENV VAR LOADED


LOADING DATA

In [6]:
Data_File_path = os.path.join("data", "hr_policy.txt")

DATA INGESTION

In [8]:
loader = TextLoader(Data_File_path, encoding="utf-8")

documents = loader.load()
print("data loaded")

print(documents)

data loaded
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDu

SPLITTING THE DOC

In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

chunks= text_splitter.split_documents(documents)
print(chunks)


[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [16]:
len(chunks)
print(chunks[6].page_content)

6. CODE OF CONDUCT
Employees are expected to maintain professionalism and respect in the workplace.
Harassment, discrimination, or any form of workplace misconduct will not be tolerated
and may result in disciplinary action, including termination.
All employees must complete an annual Code of Conduct training.


EMBED OUR DATA


In [20]:
embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")
print("embedding model is ready, name is ",embeddings_model.model_name)

embedding model is ready, name is  jina-embeddings-v2-base-en


store the date in vectorDB

In [21]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks,embeddings_model)
print("chunks are stored",vector_store.index.ntotal)

chunks are stored 9


In [22]:
test_query = "how many sick leaves employees get"

##similarity search

top_matches = vector_store.similarity_search(test_query, k=2)
print(f"Query: {test_query}\n")
for i,match in enumerate(top_matches, start=1):
    print(f"---Match {i} ---")
    print(match.page_content)

Query: how many sick leaves employees get

---Match 1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.
---Match 2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.


TOOL

In [64]:
retriever = vector_store.as_retriever(search_kwargs={"k":3}) #returns top relevant chunks
from langchain_core.tools import tool

@tool
def search_hr_policy(question: str) -> str:
    """
    Search the HR policy document for relevant information.
    """
    matching_chunks = retriever.invoke(question)

    return "\n\n".join(
        chunk.page_content for chunk in matching_chunks
    )

LLM


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0 # temp is creativity of the model from 0 to 1. but high creativity creates high randomness
)

llm.model_name

'openai/gpt-oss-120b'

In [24]:
test_response = llm.invoke("hey is learning rag hard? answer in 1 line")
print(test_response.content)

Learning RAG can be challenging at first, but with clear resources and hands‑on practice it becomes manageable.


In [26]:
from langchain.agents import create_agent


In [65]:
hr_assistant = create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt="""

    You are friendly HR assistant working for Acme crop.
    Always use the search_hr_policy tool to look up facts before answering.
    If the answer isn't in the search result, say you don't know, instead of guessing.

    """
)

print("HR assistant is ready to answer questions!")

HR assistant is ready to answer questions!


In [56]:
def ask_hr_assistant(question:str)->str:
    """Send a question to the RAG agent and print a nicely formatted answer"""
    print("=" * 60)
    print("QUESTION:", question)

    response = hr_assistant.invoke({"messages":[{"role":"user","content":question}]})

    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [59]:
import inspect

print(inspect.getsource(search_hr_policy))

def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home, probation, notice period, reimbursement, code of conduct, holidays, or exit process
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)



In [60]:
result = search_hr_policy("What is the leave policy?")

print(result)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

8. EXIT POLICY
Upon resignation or termination, employees must complete a clearance process involving
IT, Finance, and HR departments before their last working day.
Full and final settlement, including any pending reimbursements and leave encashment,
is processed within 45 days of the last working day.

7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.


In [61]:
print(hr_assistant)


In [66]:
response = hr_assistant.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Tell me about leave policies"
        }
    ]
})

print(response["messages"][-1].content)

Here’s a quick overview of Acme Crop’s leave policies as outlined in the employee handbook:

| Leave Type | Entitlement | Key Requirements |
|------------|-------------|-------------------|
| **Annual (Paid) Leave** | 20 days per calendar year (full‑time employees) | • Submit the request through the HR portal **at least 5 working days** before the intended start date.<br>• Unused days can be carried forward **up to 5 days** into the next year. |
| **Sick Leave** | 10 paid days per year | • A medical certificate is required **if the sick leave exceeds 2 consecutive days**.<br>• Sick leave is tracked separately from annual leave. |

**Additional notes**

- Both leave types are **paid** and apply to full‑time staff.  
- The HR portal is the only approved channel for submitting leave requests.  
- There is no mention of other leave categories (e.g., parental, bereavement, or unpaid leave) in the excerpt you asked about, so I can’t provide details on those.  

If you need more specific info